# 7. AIND Ephys Results Collector

Build an `AINDEPhysResultsCollectorScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysResultsCollectorTask` for each coordinate. The task itself clones [aind-ephys-results-collector](https://github.com/AllenNeuralDynamics/aind-ephys-results-collector) on first use, seeds its `data/` with the union of every previous-stage output (plus a synthetic `ecephys_<session>/` folder so the capsule's `assert len(ecephys_sessions) == 1` passes), runs `code/run_capsule.py --process-name <name>`, and copies the unified layout (`preprocessed/`, `spikesorted/`, `postprocessed/`, `curated/`, `visualization/` + `processing.json` + `data_description.json`) into the single config's `coordinate_output_root`.

## Imports and prerequisites

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import obi_one as obi

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "aind-metadata-upgrader",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv


Resolved 35 packages in 740ms
Uninstalled 2 packages in 15ms
Installed 2 packages in 5ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'aind-metadata-upgrader'], returncode=0)

## Build the scan config

In [3]:
dispatch_output_path = (Path.cwd() / "output/01_dispatch_results/0").resolve()
preprocessing_output_path = Path("/tmp/aind-ephys-preprocessing/results").resolve()
spikesort_output_path = (Path.cwd() / "output/03_spikesort_results").resolve()
postprocessing_output_path = (Path.cwd() / "output/04_postprocessing_results").resolve()
curation_output_path = (Path.cwd() / "output/05_curation_results").resolve()
visualization_output_path = (Path.cwd() / "output/06_visualization_results").resolve()

for p in (
    dispatch_output_path,
    preprocessing_output_path,
    spikesort_output_path,
    postprocessing_output_path,
    curation_output_path,
    visualization_output_path,
):
    assert p.exists(), f"{p} not found. Run earlier notebooks first."

scan_config = obi.AINDEPhysResultsCollectorScanConfig(
    initialize=obi.AINDEPhysResultsCollectorScanConfig.Initialize(
        dispatch_output_path=dispatch_output_path,
        preprocessing_output_path=preprocessing_output_path,
        spikesort_output_path=spikesort_output_path,
        postprocessing_output_path=postprocessing_output_path,
        curation_output_path=curation_output_path,
        visualization_output_path=visualization_output_path,
        process_name="sorted",
        session_name="ecephys_toy",
        subject_id="000000",
    ),
)

## Generate the grid scan and run the results-collector task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/07_aind_ephys_results_collector/grid_scan",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 17:10:51,509] INFO: Cloning https://github.com/AllenNeuralDynamics/aind-ephys-results-collector.git -> /tmp/aind-ephys-results-collector


Cloning into '/tmp/aind-ephys-results-collector'...


[2026-04-29 17:10:53,534] INFO: Seeded data dir: /tmp/aind-ephys-results-collector/data


[2026-04-29 17:10:53,536] INFO: Running python -u run_capsule.py --process-name sorted


[2026-04-29 17:10:55,248] INFO: 

COLLECTING RESULTS
Loaded session name from JSON files: session0
Copying preprocessed folders to results:
	block0_None_recording1
Copying spikesorted folders to results:
	block0_None_recording1
Copying postprocessed and curated folders to results:
	block0_None_recording1
		Remapping recording path for postprocessed to ../results/postprocessed/block0_None_recording1.zarr
	Resolving path for folder_path - ../../../../../Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/output/00_toy_example_recording
Copying visualization outputs to results:
Generating processing metadata
Generating data_description metadata
Failed to instantiate data description: 2 validation errors for DataDescription
funding_source.0.funder
  Input tag 'Allen Institute for Neural Dynamics' found using 'name' does not match any of the expected tags: 'Allen Institute', 'Chan Zuckerberg Initiative', 'MBF Bioscience', "Michael J. Fox Foundation for Parkinson's Resea

[PosixPath('../../../obi-output/07_aind_ephys_results_collector/grid_scan/AINDEPhysResultsCollectorSingleConfig')]

## Copy the per-coordinate collected results next to the notebook

In [5]:
local_results_dir = Path.cwd() / "output/07_collected_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
local_results_dir.mkdir(parents=True, exist_ok=True)

for single_config in grid_scan.single_configs:
    coord_dir = Path(single_config.coordinate_output_root)
    dest = local_results_dir / str(single_config.idx)
    dest.mkdir(parents=True, exist_ok=True)
    for entry in coord_dir.iterdir():
        target = dest / entry.name
        if target.exists():
            shutil.rmtree(target) if target.is_dir() else target.unlink()
        if entry.is_dir():
            shutil.copytree(entry, target)
        else:
            shutil.copy2(entry, target)

first = local_results_dir / "0"
if first.exists():
    for entry in first.iterdir():
        target = local_results_dir / entry.name
        if target.exists():
            shutil.rmtree(target) if target.is_dir() else target.unlink()
        if entry.is_dir():
            shutil.copytree(entry, target)
        else:
            shutil.copy2(entry, target)

for p in sorted(local_results_dir.iterdir()):
    print(" ", p.name)

  0
  curated
  data_description.json
  obi_one_coordinate.json
  postprocessed
  preprocessed
  processing.json
  spikesorted
  subject.json
  visualization
  visualization_output.json


## Inspect the collected processing.json

In [6]:
processing = json.loads((local_results_dir / "processing.json").read_text())
print("schema_version:", processing.get("schema_version"))
for pipeline in processing.get("pipelines", []):
    print("pipeline:", pipeline.get("name"))
process_names = [d.get("name") for d in processing.get("data_processes", [])]
print("data_processes:", process_names)

schema_version: 2.2.5
pipeline: AIND Ephys Pipeline
data_processes: ['Ephys preprocessing - block0_None_recording1', 'Spike sorting - block0_None_recording1', 'Ephys postprocessing - block0_None_recording1', 'Ephys curation - block0_None_recording1', 'Ephys visualization - block0_None_recording1']
